# Notebook 14 – Hardware-Lauf: Quantum Kernel SVM auf Iris

**Ziel:** Quantum Kernel SVM auf echter IBM-Hardware (`ibm_kingston`) ausführen.  
**Konfiguration:** zz_feature_map, reps=2, linear, shots=1024, n_train=30  
**Datensatz:** Iris, Features [0, 2], kanonische Pipeline  
**Ergebnis:** wird in `ergebnisse.csv` gespeichert

## 1. Imports

In [12]:
from qiskit_machine_learning.state_fidelities import ComputeUncompute
import inspect
print(inspect.signature(ComputeUncompute.__init__))

(self, sampler: 'BaseSamplerV2', *, options: 'Options | None' = None, local: 'bool' = False, pass_manager: 'PassManager | None' = None) -> 'None'


In [13]:
import numpy as np
import time
import pandas as pd
from datetime import datetime

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from qiskit.circuit.library import zz_feature_map
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

print('Imports erfolgreich')

Imports erfolgreich


## 2. IBM Quantum Verbindung

In [14]:
# API-Token hier einfügen
API_TOKEN = 'v7vNLPiYw-WbnfQI6YGZZsO1aCts-YzPzIc3wj0gSqn5'

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    token=API_TOKEN
)

backend = service.backend('ibm_kingston')
print(f'Backend: {backend.name}')
print(f'Status: {backend.status().status_msg}')

qiskit_runtime_service._discover_account:WARNING:2026-05-23 15:32:32,102: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-05-23 15:32:35,078: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-23 15:32:35,079: Using instance: open-instance, plan: open


Backend: ibm_kingston
Status: active


## 3. Kanonische Datenpipeline

In [15]:
# Iris laden – Features 0 und 2
iris = load_iris()
X = iris.data[:, [0, 2]]
y = iris.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scaler
scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# n_train=30 (QPU-Kontingent)
N_TRAIN = 30
X_train_hw = X_train[:N_TRAIN]
y_train_hw = y_train[:N_TRAIN]

print(f'X_train_hw: {X_train_hw.shape}')
print(f'X_test:     {X_test.shape}')
print(f'Klassen:    {np.unique(y_train_hw)}')

X_train_hw: (30, 2)
X_test:     (45, 2)
Klassen:    [0 1 2]


## 4. Hardware-Setup

In [16]:
# Konfiguration (beste aus NB 12)
SHOTS = 1024
OPTIMIZATION_LEVEL = 1
REPS = 2
ENTANGLEMENT = 'linear'

# Feature Map
feature_map = zz_feature_map(
    feature_dimension=2,
    reps=REPS,
    entanglement=ENTANGLEMENT
)
print(f'Feature Map Tiefe: {feature_map.depth()}')
print(f'Anzahl Qubits (logisch): {feature_map.num_qubits}')

# PassManager für Hardware-Transpilation
t0 = time.time()
pm = generate_preset_pass_manager(backend=backend, optimization_level=OPTIMIZATION_LEVEL)
qc_t = pm.run(feature_map)
circ_depth_physical = qc_t.depth()
n_physical_qubits   = qc_t.num_qubits
gate_count_physical = sum(qc_t.count_ops().values())

print(f'Logische Qubits:   {feature_map.num_qubits}')
print(f'Physische Qubits:  {n_physical_qubits}')
print(f'Schaltkreistiefe:  {circ_depth_physical}')
print(f'Gate-Anzahl:       {gate_count_physical}')

t_transpile = time.time() - t0
print(f'PassManager erstellt in {t_transpile:.2f}s')

# Hardware Sampler
sampler = Sampler(backend)
sampler.options.default_shots = SHOTS

# Fidelity + Kernel
fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity,
    max_circuits_per_job=300
)

print('Hardware-Setup abgeschlossen')

Feature Map Tiefe: 10
Anzahl Qubits (logisch): 2
Logische Qubits:   2
Physische Qubits:  156
Schaltkreistiefe:  32
Gate-Anzahl:       40
PassManager erstellt in 0.85s
Hardware-Setup abgeschlossen


## 5. Kernel-Matrix berechnen (Training)

In [17]:
print(f'Starte Kernel-Berechnung auf {backend.name}...')
print(f'Trainingspunkte: {N_TRAIN} → {N_TRAIN*(N_TRAIN+1)//2} Kernel-Evaluierungen')
print(f'Shots: {SHOTS}')
print(f'Geschätzte QPU-Zeit: ~2 Min')
print('---')

t0 = time.time()
K_train = kernel.evaluate(x_vec=X_train_hw)
t_kernel_train = time.time() - t0

print(f'Kernel-Matrix (Training) berechnet in {t_kernel_train:.2f}s')
print(f'Shape: {K_train.shape}')

Starte Kernel-Berechnung auf ibm_kingston...
Trainingspunkte: 30 → 465 Kernel-Evaluierungen
Shots: 1024
Geschätzte QPU-Zeit: ~2 Min
---
Kernel-Matrix (Training) berechnet in 146.29s
Shape: (30, 30)


## 6. SVM trainieren

In [18]:
t0 = time.time()
svm = SVC(kernel='precomputed')
svm.fit(K_train, y_train_hw)
t_fit = time.time() - t0

print(f'SVM trainiert in {t_fit:.4f}s')

SVM trainiert in 0.0113s


## 7. Kernel-Matrix berechnen (Inferenz)

In [22]:
print(np.bincount(y_train_hw))

[12  9  9]


In [19]:
print(f'Inferenz: {len(X_test)} Testpunkte × {N_TRAIN} Trainingspunkte')
print(f'= {len(X_test) * N_TRAIN} Kernel-Evaluierungen')

t0 = time.time()
K_test = kernel.evaluate(x_vec=X_test, y_vec=X_train_hw)
t_infer = time.time() - t0

print(f'Kernel-Matrix (Inferenz) berechnet in {t_infer:.2f}s')

Inferenz: 45 Testpunkte × 30 Trainingspunkte
= 1350 Kernel-Evaluierungen
Kernel-Matrix (Inferenz) berechnet in 441.52s


## 8. Evaluation

In [20]:
y_pred = svm.predict(K_test)

accuracy  = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)

t_gesamt = t_kernel_train + t_fit + t_infer
n_kernel_evals = N_TRAIN * (N_TRAIN + 1) // 2 + len(X_test) * N_TRAIN

print('=== Ergebnisse Hardware-Lauf ===')
print(f'Accuracy:  {accuracy:.4f}')
print(f'F1:        {f1:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'---')
print(f't_kernel_train: {t_kernel_train:.2f}s')
print(f't_fit:          {t_fit:.4f}s')
print(f't_infer:        {t_infer:.2f}s')
print(f't_gesamt:       {t_gesamt:.2f}s')
print(f'Kernel-Evaluierungen gesamt: {n_kernel_evals}')

=== Ergebnisse Hardware-Lauf ===
Accuracy:  0.5778
F1:        0.5720
Precision: 0.5808
Recall:    0.5778
---
t_kernel_train: 146.29s
t_fit:          0.0113s
t_infer:        441.52s
t_gesamt:       587.83s
Kernel-Evaluierungen gesamt: 1815


## 9. Ergebnis speichern

In [21]:
CSV_PATH = 'ergebnisse.csv'

result = {
    'timestamp':    datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'modell':       'Quantum Kernel SVM',
    'datensatz':    'Iris (3 Klassen, 2 Features, hardware)',
    'backend':      backend.name,
    'feature_map':  'zz_feature_map',
    'reps':         REPS,
    'entanglement': ENTANGLEMENT,
    'shots':        SHOTS,
    'n_train':      N_TRAIN,
    'n_test':       len(X_test),
    'opt_level':    OPTIMIZATION_LEVEL,
    'accuracy':     round(accuracy, 4),
    'f1':           round(f1, 4),
    'precision':    round(precision, 4),
    'recall':       round(recall, 4),
    't_kernel_train': round(t_kernel_train, 2),
    't_fit':        round(t_fit, 4),
    't_infer':      round(t_infer, 2),
    't_gesamt':     round(t_gesamt, 2),
    'circ_depth_physical': circ_depth_physical,
    'n_physical_qubits':   n_physical_qubits,
    'gate_count_physical': gate_count_physical,
    'n_kernel_evals':      n_kernel_evals,
}

df_new = pd.DataFrame([result])

try:
    df_existing = pd.read_csv(CSV_PATH)
    df_out = pd.concat([df_existing, df_new], ignore_index=True)
    print(f'Vorhandene CSV geladen ({len(df_existing)} Einträge), füge neuen hinzu.')
except FileNotFoundError:
    df_out = df_new
    print('Neue CSV wird erstellt.')

df_out.to_csv(CSV_PATH, index=False)
print(f'Gespeichert in {CSV_PATH}')
print(df_new.T)

Neue CSV wird erstellt.
Gespeichert in ergebnisse.csv
                                                          0
timestamp                               2026-05-23 15:42:24
modell                                   Quantum Kernel SVM
datensatz            Iris (3 Klassen, 2 Features, hardware)
backend                                        ibm_kingston
feature_map                                  zz_feature_map
reps                                                      2
entanglement                                         linear
shots                                                  1024
n_train                                                  30
n_test                                                   45
opt_level                                                 1
accuracy                                             0.5778
f1                                                    0.572
precision                                            0.5808
recall                                        